In [0]:
%sql
CREATE WIDGET TEXT run_month DEFAULT '2026-01';

In [0]:
%sql
USE com_edp_prd.cmpa_insights_internal_schema;

In [0]:
%sql
CREATE OR REPLACE TABLE patient360_dq_results (
run_month STRING,
check_category STRING,
check_name STRING,
status STRING, 
failed_row_count BIGINT,
total_row_count BIGINT,
failure_threshold STRING,
comments STRING,
run_timestamp TIMESTAMP
)
USING DELTA;

In [0]:
%sql
INSERT INTO patient360_dq_results
SELECT
'${run_month}',
'SNAPSHOT',
'BASE_ROW_COUNT',
'INFO',
0,
COUNT(*),
'Reference only',
'Total patients for the month',
current_timestamp()
FROM patient360;

In [0]:
%sql
INSERT INTO patient360_dq_results
SELECT
'${run_month}',
'L1_STRUCTURAL',
'ROW_COUNT_NON_ZERO',
CASE WHEN COUNT(*) > 0 THEN 'PASS' ELSE 'FAIL' END,
CASE WHEN COUNT(*) = 0 THEN COUNT(*) ELSE 0 END,
COUNT(*),
'> 0 rows',
'Ensures table is populated',
current_timestamp()
FROM patient360;

In [0]:
%sql
WITH dupes AS (
SELECT PATIENT_ID
FROM patient360
GROUP BY PATIENT_ID
HAVING COUNT(*) > 1
)
INSERT INTO patient360_dq_results
SELECT
'${run_month}',
'L1_STRUCTURAL',
'PATIENT_ID_UNIQUENESS',
CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END,
COUNT(*),
(SELECT COUNT(*) FROM patient360),
'0 duplicate PATIENT_IDs',
'Primary key uniqueness check',
current_timestamp()
FROM dupes;

In [0]:
%sql
INSERT INTO patient360_dq_results
SELECT
  :run_month AS run_month,
  'L1_STRUCTURAL' AS check_category,
  'LATEST_CLAIM_FRESHNESS' AS check_name,
  CASE 
    WHEN MAX(latest_claim_date) >= add_months(current_date(), -2)
    THEN 'PASS'
    ELSE 'WARN'
  END AS status,
  0 AS failed_row_count,
  COUNT(*) AS total_row_count,
  'Max claim within last 60 days' AS failure_threshold,
  CONCAT('Max latest_claim_date = ', CAST(MAX(latest_claim_date) AS STRING)) AS comments,
  current_timestamp() AS run_timestamp
FROM patient360;


In [0]:
%sql
WITH nulls AS (
SELECT *
FROM patient360
WHERE PATIENT_ID IS NULL
OR incidence_date IS NULL
OR latest_claim_date IS NULL
)
INSERT INTO patient360_dq_results
SELECT distinct
'${run_month}',
'L2_COLUMN',
'MANDATORY_FIELDS_NOT_NULL',
CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END,
COUNT(*),
(SELECT COUNT(*) FROM patient360),
'No NULLs in critical fields- PATIENT_ID, incidence_date, latest_claim_date',
'Checks core identifiers and dates',
current_timestamp()
FROM nulls;

In [0]:
%sql
WITH invalid_gender AS (
SELECT *
FROM patient360
WHERE PATIENT_GENDER NOT IN ('M','F','U') OR PATIENT_GENDER IS NULL
)
INSERT INTO patient360_dq_results
SELECT
'${run_month}',
'L2_COLUMN',
'PATIENT_GENDER_DOMAIN',
CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'WARN' END,
COUNT(*),
(SELECT COUNT(*) FROM patient360),
'Allowed values: M/F/U',
'Gender domain validation',
current_timestamp()
FROM invalid_gender;

In [0]:
%sql
WITH invalid_dx_tx AS (
SELECT *
FROM patient360
WHERE first_tx_after_diagnosis < incidence_date
)
INSERT INTO patient360_dq_results
SELECT distinct
'${run_month}',
'L3_BUSINESS',
'DX_BEFORE_TX',
CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END,
COUNT(*),
(SELECT COUNT(*) FROM patient360),
'first_tx_after_diagnosis < Dx date',
'Treatment cannot precede diagnosis',
current_timestamp()
FROM invalid_dx_tx;

In [0]:
%sql
WITH invalid_dates AS (
SELECT *
FROM patient360
WHERE latest_treatment_date > latest_claim_date
)
INSERT INTO patient360_dq_results
SELECT distinct
'${run_month}',
'L3_BUSINESS',
'LATEST_TX_AFTER_CLAIM',
CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END,
COUNT(*),
(SELECT COUNT(*) FROM patient360),
'latest_treatment_date_5yr > latest_claim_date',
'Treatment date exceeds last claim',
current_timestamp()
FROM invalid_dates;

In [0]:
%sql
INSERT INTO patient360_dq_results
SELECT distinct
'${run_month}',
'L4_TREND',
CONCAT('SEVERITY_DIST_', severity),
'INFO',
0,
COUNT(*),
'Tracking only',
'Monthly severity distribution snapshot',
current_timestamp()
FROM patient360
GROUP BY severity;

In [0]:
%sql
SELECT distinct
run_month,
check_category,
COUNT(*) AS total_checks,
SUM(CASE WHEN status = 'FAIL' THEN 1 ELSE 0 END) AS failed_checks,
SUM(CASE WHEN status = 'WARN' THEN 1 ELSE 0 END) AS warning_checks
FROM patient360_dq_results
WHERE run_month = '${run_month}'
GROUP BY run_month, check_category
ORDER BY check_category;

In [0]:
%sql
SELECT distinct *
FROM patient360_dq_results
WHERE run_month = '${run_month}'
AND status IN ('FAIL','WARN')
ORDER BY check_category, check_name;

In [0]:
%sql
SELECT distinct *
FROM patient360_dq_results